# FER2013 Emotion CNN in PyTorch

A step-by-step PyTorch port of the Keras example. The notebook follows the same flow: overview, data, model, training, evaluation, and one prediction.

## Import libraries and shared data helpers

The model is defined below in this notebook, as in the Keras example. The script in `src/` contains the reusable command-line version.

In [ ]:
import sys
from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [
    Path.cwd() / "BTL03-ConvertKerasToPytorch" / "src",
    Path.cwd() / "src",
    Path("/content"),
]
SRC_DIR = next((p for p in candidates if (p / "fer2013_emotion_cnn.py").is_file()), None)
if SRC_DIR is None:
    raise FileNotFoundError("Could not find src/fer2013_emotion_cnn.py. Upload this script to Colab or run from the repository root.")
sys.path.insert(0, str(SRC_DIR.resolve()))
from fer2013_emotion_cnn import (
    CLASS_NAMES, load_fer_csv, make_loader,
    run_epoch, set_seed, save_history_plot,
)

num_classes = len(CLASS_NAMES)
print(CLASS_NAMES)

## (1) Data

Each CSV row contains 2,304 grayscale pixels for a 48x48 image, followed by 7 one-hot class labels. PublicTest is used as validation, matching the source notebook.

In [ ]:
TRAIN_CSV = Path("fer2013_training_onehot.csv")
PUBLICTEST_CSV = Path("fer2013_publictest_onehot.csv")

x_training, y_training = load_fer_csv(TRAIN_CSV)
x_publictest, y_publictest = load_fer_csv(PUBLICTEST_CSV)
print("Train:", tuple(x_training.shape), tuple(y_training.shape))
print("PublicTest:", tuple(x_publictest.shape), tuple(y_publictest.shape))
print("Input layout: NCHW; pixel range is preserved from the CSV (normally 0..255).")

## (2) Declare model

This directly maps Keras `Conv2D`, `MaxPooling2D`, `Dropout`, `Flatten`, and `Dense` layers to PyTorch. PyTorch uses NCHW; we transpose to NHWC before flattening to preserve Keras feature order.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EmotionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1_1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)  # Keras padding="same"
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=0) # Keras padding="valid"
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv_dropout1 = nn.Dropout(p=0.25)

        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=0)
        self.conv2_3 = nn.Conv2d(128, 218, kernel_size=3, padding=1)
        self.conv2_4 = nn.Conv2d(218, 218, kernel_size=3, padding=0)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv_dropout2 = nn.Dropout(p=0.25)

        self.fc1 = nn.Linear(218 * 9 * 9, 1024)
        self.fc_dropout1 = nn.Dropout(p=0.5)
        self.fc3_7ways = nn.Linear(1024, len(CLASS_NAMES))

        # Match Keras default Glorot-uniform kernels and zero biases.
        for layer in self.modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, images):
        x = F.relu(self.conv1_1(images))
        x = F.relu(self.conv1_2(x))
        x = self.conv_dropout1(self.pool1(x))
        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = F.relu(self.conv2_3(x))
        x = F.relu(self.conv2_4(x))
        x = self.conv_dropout2(self.pool2(x))
        # Keras Flatten traverses NHWC; preserve the same feature order.
        x = x.permute(0, 2, 3, 1).contiguous().view(x.shape[0], -1)
        x = self.fc_dropout1(F.relu(self.fc1(x)))
        return self.fc3_7ways(x)  # logits; CrossEntropyLoss includes log-softmax

model = EmotionCNN().to(device)
print(model)
print("Device:", device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## (3) Train defined model

CrossEntropyLoss takes class indices instead of one-hot labels. The notebook keeps Adam at learning rate 0.0001 and batch size 128. The utilities variant adds augmentation and early stopping with patience 5.

In [ ]:
VARIANT = "basic"
USE_AUGMENTATION = False
EPOCHS = 15
PATIENCE = 5
BATCH_SIZE = 128
LEARNING_RATE = 1e-4
SEED = 42
RESULT_DIR = Path("fer2013_pytorch_results") / VARIANT

set_seed(SEED)
train_loader = make_loader(x_training, y_training, BATCH_SIZE, shuffle=True, seed=SEED)
publictest_loader = make_loader(x_publictest, y_publictest, BATCH_SIZE, shuffle=False, seed=SEED)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
best_val_loss = float("inf")
wait = 0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(
        model, train_loader, device, optimizer, use_augmentation=USE_AUGMENTATION
    )
    val_loss, val_accuracy = run_epoch(model, publictest_loader, device)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    print(f"{epoch:03d}/{EPOCHS} train_loss={train_loss:.4f} train_acc={train_accuracy:.4f} "
          f"val_loss={val_loss:.4f} val_acc={val_accuracy:.4f}")

    if USE_AUGMENTATION:
        if val_loss < best_val_loss:
            best_val_loss, best_epoch, wait = val_loss, epoch, 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stopping at epoch {epoch}; best val_loss was at epoch {best_epoch}.")
                break

print("Epochs completed:", len(history["train_loss"]))


## (4) Evaluate trained model

The source workflow reports metrics on PublicTest, which was also used for validation during training. This is not an independent test split.

In [ ]:
publictest_loss, publictest_accuracy = run_epoch(model, publictest_loader, device)
print("PublicTest loss:", publictest_loss)
print("PublicTest accuracy:", publictest_accuracy)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = RESULT_DIR / "fer2013_final.pt"
torch.save({
    "state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "variant": VARIANT,
    "final_epoch": len(history["train_loss"]),
}, checkpoint_path)
(RESULT_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
save_history_plot(history, RESULT_DIR / "training_history.png", f"FER2013 CNN - {VARIANT}")
print("Saved:", checkpoint_path.resolve())

## (5) Predict one sample

In [ ]:
index = 0
model.eval()
with torch.no_grad():
    probabilities = torch.softmax(model(x_publictest[index:index + 1].to(device)), dim=1)[0].cpu()
predicted_index = int(probabilities.argmax())
true_index = int(y_publictest[index])

plt.imshow(x_publictest[index, 0].numpy(), cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.title(f"True: {CLASS_NAMES[true_index]} | Predicted: {CLASS_NAMES[predicted_index]}")
plt.show()
print("Probabilities:", dict(zip(CLASS_NAMES, probabilities.tolist())))